In [3]:
import os
import random
import urllib.request
import numpy as np
import cv2
import pickle
import tensorflow as tf
from tensorflow import keras
from keras_facenet import FaceNet
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from sklearn.preprocessing import normalize


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

In [ ]:
# 1. Download MediaPipe Face Detector Model

url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
output_path = "detector.tflite"

if not os.path.exists(output_path):
    print("Downloading MediaPipe Face Detector model...")
    urllib.request.urlretrieve(url, output_path)


In [ ]:
# Setup MediaPipe Face Detector
base_options = python.BaseOptions(model_asset_path='detector.tflite')
options = vision.FaceDetectorOptions(base_options=base_options)
detector = vision.FaceDetector.create_from_options(options)

In [ ]:
# 2. Crop Faces from Dataset using MediaPipe
dataset_base = r"C:\Users\AYA\Downloads\data set"

persons = [
    d for d in os.listdir(dataset_base) 
    if os.path.isdir(os.path.join(dataset_base, d)) and not d.endswith("_cropped") and not d.endswith("test data")
]

print(f"Found persons: {persons}")

Found persons: ['Ashraqat', 'person2']


In [ ]:
for person in persons:
    input_folder = os.path.join(dataset_base, person)
    output_folder = os.path.join(dataset_base, f"{person}_cropped")
    os.makedirs(output_folder, exist_ok=True)
    
    if not os.path.exists(input_folder):
        continue
        
    image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Processing images for: {person} ({len(image_files)} images)...")
    
    for filename in image_files:
        image_path = os.path.join(input_folder, filename)
        image_cv = cv2.imread(image_path)
        if image_cv is None:
            continue
            
        image_mp = mp.Image.create_from_file(image_path)
        detection_result = detector.detect(image_mp)
        
        if detection_result.detections:
            for detection in detection_result.detections:
                bbox = detection.bounding_box
                x, y, w, h = max(0, bbox.origin_x), max(0, bbox.origin_y), bbox.width, bbox.height
                
                margin = int(0.1 * min(w, h))
                x_start = max(0, x - margin)
                y_start = max(0, y - margin)
                x_end = min(image_cv.shape[1], x + w + margin)
                y_end = min(image_cv.shape[0], y + h + margin)
                
                face_crop = image_cv[y_start:y_end, x_start:x_end]
                
                if face_crop.size > 0:
                    face_resized = cv2.resize(face_crop, (160, 160))
                    save_path = os.path.join(output_folder, f"crop_{filename}")
                    cv2.imwrite(save_path, face_resized)

print("All faces cropped and saved successfully!")

In [ ]:
# Model 
embedder = FaceNet()
ENCODINGS_FILE = "encodings.pickle"

In [ ]:
all_face_arrays = []
all_labels = []

for person in persons:
    cropped_folder = os.path.join(dataset_base, f"{person}_cropped")
    
    if os.path.exists(cropped_folder):
        files = [f for f in os.listdir(cropped_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"Extracting encodings for: {person}...")
        
        for filename in files:
            image_path = os.path.join(cropped_folder, filename)
            img = cv2.imread(image_path)
            if img is None:
                continue
            
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, (160, 160))
            
           
            encoding = embedder.embeddings([img_resized])[0]
            
            all_face_arrays.append(encoding)
            all_labels.append(person)

all_face_arrays = np.array(all_face_arrays)
print(f"\nExtracted total of {len(all_face_arrays)} encodings.")

with open(ENCODINGS_FILE, "wb") as f:
    pickle.dump({"encodings": all_face_arrays, "names": all_labels}, f)
print(f"Saved encodings to {ENCODINGS_FILE}")


Extracting encodings for: Ashraqat...
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━

In [ ]:
# 4. Train & Evaluate KNN Model
X_train, X_test, y_train, y_test = train_test_split(
    all_face_arrays, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)


    
X_train_norm = normalize(X_train, norm='l2')
X_test_norm = normalize(X_test, norm='l2')


knn_clf = KNeighborsClassifier(n_neighbors=3, metric='cosine', weights='distance')
knn_clf.fit(X_train_norm, y_train)


y_pred = knn_clf.predict(X_test_norm)


In [ ]:
print("\n--- Classification Report ---")

print(classification_report(y_test, y_pred, zero_division=0))


--- Classification Report ---
              precision    recall  f1-score   support

    Ashraqat       1.00      0.83      0.91         6
     person2       0.86      1.00      0.92         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12

